In [ ]:
# CONSOLIDATED COLAB NOTEBOOK (TRAIN + TEST ALL PRIMARY MODELS)
# =============================================================
#
# Required columns (use what applies; notebook handles missing optional columns):
# created_at, summary, description,
# final_category, final_priority, final_emotion,
# ticket_context, log_line, log_relevant,             # for log reranker (pair rows)
# retrieval_query, retrieval_doc, retrieval_group_id, retrieval_relevant,  # for retrieval eval
# incident_in_next_30m,                               # for incident prediction
# comments_text, top_error_lines, affected_users, downtime_minutes, error_count,
# fatal_count, timeout_count, auth_error_count, env, service
# =============================================================

# =============================================================
# PUBLIC DATASET LOADER / NORMALIZER
# Adds public datasets for training/testing and maps them into
# the notebook's expected schema.
# =============================================================

!pip -q install datasets huggingface_hub pyarrow gdown kagglehub

import os
import json
import random
import numpy as np
import pandas as pd
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE = "/content/drive/MyDrive/jsm_ai_ops"
DATA_DIR = f"{BASE}/data"
ART = f"{BASE}/artifacts_all"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ART, exist_ok=True)

PUBLIC_DATA_CSV = f"{DATA_DIR}/tickets_public_combined.csv"

TARGET_COLUMNS = [
    "created_at", "summary", "description",
    "final_category", "final_priority", "final_emotion",
    "ticket_context", "log_line", "log_relevant",
    "retrieval_query", "retrieval_doc", "retrieval_group_id", "retrieval_relevant",
    "incident_in_next_30m",
    "comments_text", "top_error_lines",
    "affected_users", "downtime_minutes", "error_count",
    "fatal_count", "timeout_count", "auth_error_count",
    "env", "service"
]

def empty_frame():
    return pd.DataFrame(columns=TARGET_COLUMNS)

def ensure_schema(df):
    df = df.copy()
    for c in TARGET_COLUMNS:
        if c not in df.columns:
            if c in [
                "log_relevant", "retrieval_relevant", "incident_in_next_30m",
                "affected_users", "downtime_minutes", "error_count",
                "fatal_count", "timeout_count", "auth_error_count"
            ]:
                df[c] = 0
            else:
                df[c] = ""
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    if df["created_at"].isna().all():
        df["created_at"] = pd.Timestamp.utcnow()
    else:
        df["created_at"] = df["created_at"].fillna(pd.Timestamp.utcnow())
    return df[TARGET_COLUMNS]

# -------------------------------------------------------------
# 1) TriageIQ: category + urgency + sentiment
# Maps:
#   category -> final_category
#   urgency  -> final_priority (proxy map)
#   sentiment -> final_emotion
# -------------------------------------------------------------
def load_triageiq(max_rows=None):
    try:
        ds = load_dataset("coldstart88/triageiq-dataset")
        split_name = list(ds.keys())[0]
        pdf = ds[split_name].to_pandas()

        # Defensive column mapping
        cols = {c.lower(): c for c in pdf.columns}
        text_col = cols.get("text") or cols.get("ticket") or cols.get("content")
        cat_col = cols.get("category")
        urg_col = cols.get("urgency")
        emo_col = cols.get("sentiment")

        if text_col is None:
            raise ValueError(f"TriageIQ text column not found. Columns: {list(pdf.columns)}")

        out = empty_frame()
        out["summary"] = pdf[text_col].astype(str).str.slice(0, 160)
        out["description"] = pdf[text_col].astype(str)
        out["comments_text"] = ""
        out["top_error_lines"] = ""
        out["env"] = "unknown"
        out["service"] = "support"

        if cat_col:
            out["final_category"] = pdf[cat_col].astype(str)

        if urg_col:
            urgency_map = {
                "low": "P4",
                "medium": "P3",
                "high": "P2",
                "critical": "P1"
            }
            out["final_priority"] = (
                pdf[urg_col].astype(str).str.lower().map(urgency_map).fillna("P3")
            )

        if emo_col:
            emo_map = {
                "negative": "frustrated",
                "neutral": "neutral",
                "positive": "neutral"
            }
            out["final_emotion"] = (
                pdf[emo_col].astype(str).str.lower().map(emo_map).fillna("neutral")
            )

        out["created_at"] = pd.Timestamp.utcnow()

        if max_rows:
            out = out.sample(min(max_rows, len(out)), random_state=SEED)

        return ensure_schema(out)

    except Exception as e:
        print("Skipping TriageIQ:", e)
        return empty_frame()

# -------------------------------------------------------------
# 2) Support Ticket Intents: augment text/category
# Maps intent-like label -> final_category
# -------------------------------------------------------------
def load_support_ticket_intents(max_rows=None):
    try:
        ds = load_dataset("irongateprd/support-ticket-intents")
        split_name = list(ds.keys())[0]
        pdf = ds[split_name].to_pandas()

        cols = {c.lower(): c for c in pdf.columns}
        text_col = cols.get("text") or cols.get("ticket") or cols.get("utterance") or cols.get("content")
        label_col = cols.get("label") or cols.get("intent") or cols.get("category")

        if text_col is None:
            raise ValueError(f"Support-ticket-intents text column not found. Columns: {list(pdf.columns)}")

        out = empty_frame()
        out["summary"] = pdf[text_col].astype(str).str.slice(0, 160)
        out["description"] = pdf[text_col].astype(str)
        out["comments_text"] = ""
        out["top_error_lines"] = ""
        out["created_at"] = pd.Timestamp.utcnow()
        out["env"] = "unknown"
        out["service"] = "support"

        if label_col:
            out["final_category"] = pdf[label_col].astype(str)

        if max_rows:
            out = out.sample(min(max_rows, len(out)), random_state=SEED)

        return ensure_schema(out)

    except Exception as e:
        print("Skipping support-ticket-intents:", e)
        return empty_frame()

# -------------------------------------------------------------
# 3) BEIR: retrieval eval rows
# Produces:
#   retrieval_query, retrieval_doc, retrieval_group_id, retrieval_relevant
# Uses a small BEIR subset if available.
# -------------------------------------------------------------
def load_beir_subset(dataset_name="msmarco", max_queries=500):
    try:
        # Hugging Face BEIR layout can vary by subset; this block is defensive.
        ds = load_dataset("BeIR/beir", dataset_name)

        available = list(ds.keys())
        print("BEIR splits:", available)

        # Try common split names
        corpus_split = "corpus" if "corpus" in ds else available[0]
        queries_split = "queries" if "queries" in ds else None
        qrels_split = "qrels" if "qrels" in ds else None

        if queries_split is None or qrels_split is None:
            raise ValueError("BEIR subset missing expected queries/qrels splits in this loader.")

        corpus_df = ds[corpus_split].to_pandas()
        queries_df = ds[queries_split].to_pandas()
        qrels_df = ds[qrels_split].to_pandas()

        # Normalize likely field names
        corpus_cols = {c.lower(): c for c in corpus_df.columns}
        queries_cols = {c.lower(): c for c in queries_df.columns}
        qrels_cols = {c.lower(): c for c in qrels_df.columns}

        doc_id_col = corpus_cols.get("_id") or corpus_cols.get("id") or corpus_cols.get("doc_id")
        text_col = corpus_cols.get("text") or corpus_cols.get("contents")
        title_col = corpus_cols.get("title")

        qid_col = queries_cols.get("_id") or queries_cols.get("id") or queries_cols.get("query_id")
        query_col = queries_cols.get("text") or queries_cols.get("query")

        qrels_qid = qrels_cols.get("query-id") or qrels_cols.get("query_id") or qrels_cols.get("qid")
        qrels_docid = qrels_cols.get("corpus-id") or qrels_cols.get("corpus_id") or qrels_cols.get("doc_id")
        qrels_score = qrels_cols.get("score") or qrels_cols.get("relevance")

        if not all([doc_id_col, text_col, qid_col, query_col, qrels_qid, qrels_docid]):
            raise ValueError("Could not identify BEIR columns.")

        if title_col:
            corpus_df["joined_doc"] = corpus_df[title_col].fillna("").astype(str) + " " + corpus_df[text_col].fillna("").astype(str)
        else:
            corpus_df["joined_doc"] = corpus_df[text_col].fillna("").astype(str)

        corpus_map = corpus_df[[doc_id_col, "joined_doc"]].drop_duplicates()
        query_map = queries_df[[qid_col, query_col]].drop_duplicates()

        merged = qrels_df.merge(query_map, left_on=qrels_qid, right_on=qid_col, how="inner")
        merged = merged.merge(corpus_map, left_on=qrels_docid, right_on=doc_id_col, how="inner")

        if qrels_score:
            merged["label"] = (pd.to_numeric(merged[qrels_score], errors="coerce").fillna(0) > 0).astype(int)
        else:
            merged["label"] = 1

        merged = merged.rename(columns={
            query_col: "retrieval_query",
            "joined_doc": "retrieval_doc",
            qrels_qid: "retrieval_group_id"
        })

        merged = merged[["retrieval_query", "retrieval_doc", "retrieval_group_id", "label"]].copy()
        merged["retrieval_relevant"] = merged["label"]
        merged.drop(columns=["label"], inplace=True)

        if max_queries:
            keep_q = merged["retrieval_group_id"].astype(str).drop_duplicates().head(max_queries)
            merged = merged[merged["retrieval_group_id"].astype(str).isin(set(keep_q))]

        out = empty_frame()
        out["created_at"] = pd.Timestamp.utcnow()
        out["retrieval_query"] = merged["retrieval_query"].astype(str)
        out["retrieval_doc"] = merged["retrieval_doc"].astype(str)
        out["retrieval_group_id"] = merged["retrieval_group_id"].astype(str)
        out["retrieval_relevant"] = merged["retrieval_relevant"].astype(int)
        out["service"] = "retrieval"
        out["env"] = "benchmark"

        return ensure_schema(out)

    except Exception as e:
        print("Skipping BEIR subset:", e)
        return empty_frame()

# -------------------------------------------------------------
# 4) Optional local CSV merge
# Keeps your existing tickets.csv if present
# -------------------------------------------------------------
def load_local_csv(path):
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        return empty_frame()
    try:
        pdf = pd.read_csv(path)
    except Exception:
        try:
            pdf = pd.read_csv(path, sep=";")
        except Exception:
            pdf = pd.read_csv(path, sep="\t")
    return ensure_schema(pdf)

# -------------------------------------------------------------
# Build combined dataset
# -------------------------------------------------------------
LOCAL_CSV = f"{DATA_DIR}/tickets.csv"

parts = [
    load_local_csv(LOCAL_CSV),
    load_triageiq(max_rows=5000),
    load_support_ticket_intents(max_rows=5000),
    load_beir_subset(dataset_name="msmarco", max_queries=300),
]

df = pd.concat(parts, ignore_index=True)
df = ensure_schema(df)
df = df.drop_duplicates(subset=[
    "summary", "description", "final_category", "final_priority",
    "final_emotion", "retrieval_query", "retrieval_doc", "log_line"
]).reset_index(drop=True)

df.to_csv(PUBLIC_DATA_CSV, index=False)

print("Combined dataset shape:", df.shape)
print("Saved:", PUBLIC_DATA_CSV)
print("Columns:", df.columns.tolist())

print("\nNon-empty label coverage:")
for c in ["final_category", "final_priority", "final_emotion", "retrieval_query", "log_line", "incident_in_next_30m"]:
    if c in df.columns:
        if df[c].dtype == object:
            n = int((df[c].astype(str).str.len() > 0).sum())
        else:
            n = int(df[c].notna().sum())
        print(f"{c}: {n}")
# ----------------------------
# Basic cleanup
# ----------------------------
def ensure_col(col, default):
    if col not in df.columns:
        df[col] = default
    df[col] = df[col].fillna(default)

for c in ["summary","description","comments_text","top_error_lines","env","service",
          "final_category","final_priority","final_emotion",
          "ticket_context","log_line","retrieval_query","retrieval_doc","retrieval_group_id"]:
    ensure_col(c, "")

for c in ["affected_users","downtime_minutes","error_count","fatal_count","timeout_count","auth_error_count",
          "log_relevant","retrieval_relevant","incident_in_next_30m"]:
    if c not in df.columns:
        df[c] = 0
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

df["created_at"] = pd.to_datetime(df.get("created_at", pd.Timestamp.utcnow()), errors="coerce")
df = df.dropna(subset=["created_at"]).sort_values("created_at").reset_index(drop=True)

# Strict priority filtering
ALLOWED_P = {"P1","P2","P3","P4","P5"}
df["final_priority"] = df["final_priority"].astype(str).str.upper().str.strip()
df = df[(df["final_priority"] == "") | (df["final_priority"].isin(ALLOWED_P))].copy()

def time_split(frame, train=0.70, val=0.15):
    n = len(frame)
    i1, i2 = int(n*train), int(n*(train+val))
    return frame.iloc[:i1].copy(), frame.iloc[i1:i2].copy(), frame.iloc[i2:].copy()

train_df, val_df, test_df = time_split(df)

print("Total:", len(df), "Train/Val/Test:", len(train_df), len(val_df), len(test_df))

summary_metrics = {}


Skipping support-ticket-intents: invalid literal for int() with base 10: 'billing'
Skipping BEIR subset: Dataset scripts are no longer supported, but found beir.py
Combined dataset shape: (1398, 24)
Saved: /content/drive/MyDrive/jsm_ai_ops/data/tickets_public_combined.csv
Columns: ['created_at', 'summary', 'description', 'final_category', 'final_priority', 'final_emotion', 'ticket_context', 'log_line', 'log_relevant', 'retrieval_query', 'retrieval_doc', 'retrieval_group_id', 'retrieval_relevant', 'incident_in_next_30m', 'comments_text', 'top_error_lines', 'affected_users', 'downtime_minutes', 'error_count', 'fatal_count', 'timeout_count', 'auth_error_count', 'env', 'service']

Non-empty label coverage:
final_category: 1398
final_priority: 1398
final_emotion: 1398
retrieval_query: 1398
log_line: 1398
incident_in_next_30m: 1398
Total: 1398 Train/Val/Test: 978 210 210


/tmp/ipykernel_25557/2073232166.py:297: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(parts, ignore_index=True)


In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import torch

from datasets import Dataset


from sklearn.metrics import f1_score, accuracy_score, average_precision_score
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
!pip -q install lightgbm
import lightgbm as lgb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [ ]:
# # ============================================================
# # 1) CATEGORY MODEL — DeBERTa-v3-base (with fallback)
# # Compatible with newer transformers Trainer API
# # ============================================================


# cat_df = df[df["final_category"].astype(str).str.len() > 0].copy()

# MIN_DEEP_TRAIN_ROWS = 200
# MIN_LIGHT_TRAIN_ROWS = 30   # fallback threshold

# def mk_text(x):
#     return (
#         str(x.get("summary", "")) + " [SEP] " +
#         str(x.get("description", "")) + " [SEP] " +
#         str(x.get("comments_text", "")) + " [SEP] " +
#         str(x.get("top_error_lines", ""))
#     )

# def build_trainer(model, args, train_dataset, eval_dataset, tok, compute_metrics):
#     return Trainer(
#         model=model,
#         args=args,
#         train_dataset=train_dataset,
#         eval_dataset=eval_dataset,
#         data_collator=DataCollatorWithPadding(tokenizer=tok),
#         compute_metrics=compute_metrics
#     )

# if len(cat_df) >= MIN_DEEP_TRAIN_ROWS:
#     tr, va, te = time_split(cat_df)

#     tr["text"] = tr.apply(mk_text, axis=1)
#     va["text"] = va.apply(mk_text, axis=1)
#     te["text"] = te.apply(mk_text, axis=1)

#     le_cat = LabelEncoder()
#     y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
#     y_va = le_cat.transform(va["final_category"].astype(str))
#     y_te = le_cat.transform(te["final_category"].astype(str))

#     ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": y_tr}))
#     ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": y_va}))
#     ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": y_te}))

#     model_name = "microsoft/deberta-v3-base"
#     tok = AutoTokenizer.from_pretrained(model_name)

#     def tok_fn(b):
#         return tok(b["text"], truncation=True, max_length=384)

#     ds_tr = ds_tr.map(tok_fn, batched=True)
#     ds_va = ds_va.map(tok_fn, batched=True)
#     ds_te = ds_te.map(tok_fn, batched=True)

#     model = AutoModelForSequenceClassification.from_pretrained(
#         model_name,
#         num_labels=len(le_cat.classes_)
#     )

#     def comp(eval_pred):
#         logits, labels = eval_pred
#         pred = np.argmax(logits, axis=-1)
#         return {
#             "macro_f1": f1_score(labels, pred, average="macro"),
#             "accuracy": accuracy_score(labels, pred)
#         }

#     out_dir = f"{ART}/classification_deberta"
#     args = TrainingArguments(
#         output_dir=out_dir,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         learning_rate=2e-5,
#         per_device_train_batch_size=8,
#         per_device_eval_batch_size=16,
#         num_train_epochs=3,
#         weight_decay=0.01,
#         warmup_steps=10,
#         fp16=torch.cuda.is_available(),
#         load_best_model_at_end=True,
#         metric_for_best_model="macro_f1",
#         save_total_limit=2,
#         report_to="none"
#     )

#     trainer = build_trainer(
#         model=model,
#         args=args,
#         train_dataset=ds_tr,
#         eval_dataset=ds_va,
#         tok=tok,
#         compute_metrics=comp
#     )

#     trainer.train()
#     pred = trainer.predict(ds_te)
#     yhat = np.argmax(pred.predictions, axis=-1)
#     m = f1_score(y_te, yhat, average="macro")
#     summary_metrics["classification_macro_f1"] = float(m)

#     trainer.save_model(out_dir)
#     tok.save_pretrained(out_dir)
#     joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
#     print("Classification Macro-F1:", round(m, 4))

# elif len(cat_df) >= MIN_LIGHT_TRAIN_ROWS:
#     from sklearn.pipeline import Pipeline
#     from sklearn.feature_extraction.text import TfidfVectorizer
#     from sklearn.linear_model import LogisticRegression

#     tr, va, te = time_split(cat_df)

#     tr["text"] = tr.apply(mk_text, axis=1)
#     va["text"] = va.apply(mk_text, axis=1)
#     te["text"] = te.apply(mk_text, axis=1)

#     le_cat = LabelEncoder()
#     y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
#     y_te = le_cat.transform(te["final_category"].astype(str))

#     clf = Pipeline([
#         ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
#         ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))
#     ])

#     clf.fit(tr["text"], y_tr)
#     yhat = clf.predict(te["text"])
#     m = f1_score(y_te, yhat, average="macro")
#     summary_metrics["classification_macro_f1"] = float(m)

#     out_dir = f"{ART}/classification_tfidf_lr"
#     os.makedirs(out_dir, exist_ok=True)
#     joblib.dump(clf, f"{out_dir}/model.joblib")
#     joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
#     print("Fallback Classification (TF-IDF+LR) Macro-F1:", round(m, 4))

# else:
#     summary_metrics["classification_macro_f1"] = None
#     print(
#         f"Skipped supervised classification (labeled rows={len(cat_df)}). "
#         f"Use rule-based/zero-shot inference fallback at runtime."
#     )

# ============================================================
# 1) CATEGORY MODEL — DeBERTa-v3-base (with fallback)
# Updated for newer transformers + XLA-safe settings
# ============================================================

# Make sure these imports were run earlier:
# from datasets import Dataset
# from sklearn.preprocessing import LabelEncoder
# from sklearn.metrics import f1_score, accuracy_score
# from transformers import (
#     AutoTokenizer, AutoModelForSequenceClassification,
#     TrainingArguments, Trainer, DataCollatorWithPadding
# )

cat_df = df[df["final_category"].astype(str).str.len() > 0].copy()

MIN_DEEP_TRAIN_ROWS = 200
MIN_LIGHT_TRAIN_ROWS = 30   # fallback threshold

def mk_text(x):
    return (
        str(x.get("summary", "")) + " [SEP] " +
        str(x.get("description", "")) + " [SEP] " +
        str(x.get("comments_text", "")) + " [SEP] " +
        str(x.get("top_error_lines", ""))
    )

def is_xla_runtime():
    try:
        import torch_xla  # noqa: F401
        return True
    except Exception:
        return False

USE_FP16 = torch.cuda.is_available() and not is_xla_runtime()
USE_BF16 = False

def build_trainer(model, args, train_dataset, eval_dataset, tok, compute_metrics):
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=compute_metrics
    )

def build_args(out_dir, train_bs=8, eval_bs=16, epochs=3, lr=2e-5, metric="macro_f1"):
    return TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        weight_decay=0.01,
        warmup_steps=10,
        fp16=USE_FP16,
        bf16=USE_BF16,
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model=metric,
        save_total_limit=2,
        report_to="none"
    )

print("Category rows:", len(cat_df), "| CUDA:", torch.cuda.is_available(), "| XLA:", is_xla_runtime(), "| fp16:", USE_FP16)

if len(cat_df) >= MIN_DEEP_TRAIN_ROWS:
    tr, va, te = time_split(cat_df)

    tr["text"] = tr.apply(mk_text, axis=1)
    va["text"] = va.apply(mk_text, axis=1)
    te["text"] = te.apply(mk_text, axis=1)

    le_cat = LabelEncoder()
    y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
    y_va = le_cat.transform(va["final_category"].astype(str))
    y_te = le_cat.transform(te["final_category"].astype(str))

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": y_tr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": y_va}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": y_te}))

    model_name = "microsoft/deberta-v3-base"
    tok = AutoTokenizer.from_pretrained(model_name)

    def tok_fn(batch):
        return tok(batch["text"], truncation=True, max_length=384)

    ds_tr = ds_tr.map(tok_fn, batched=True)
    ds_va = ds_va.map(tok_fn, batched=True)
    ds_te = ds_te.map(tok_fn, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(le_cat.classes_)
    )

    def comp(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/classification_deberta"
    args = build_args(
        out_dir=out_dir,
        train_bs=8,
        eval_bs=16,
        epochs=3,
        lr=2e-5,
        metric="macro_f1"
    )

    trainer = build_trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        tok=tok,
        compute_metrics=comp
    )

    trainer.train()
    pred = trainer.predict(ds_te)
    yhat = np.argmax(pred.predictions, axis=-1)
    m = f1_score(y_te, yhat, average="macro")
    summary_metrics["classification_macro_f1"] = float(m)

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
    print("Classification Macro-F1:", round(m, 4))

elif len(cat_df) >= MIN_LIGHT_TRAIN_ROWS:
    # -------------------------------
    # Fallback: TF-IDF + LogisticRegression
    # -------------------------------
    from sklearn.pipeline import Pipeline
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression

    tr, va, te = time_split(cat_df)

    tr["text"] = tr.apply(mk_text, axis=1)
    va["text"] = va.apply(mk_text, axis=1)
    te["text"] = te.apply(mk_text, axis=1)

    le_cat = LabelEncoder()
    y_tr = le_cat.fit_transform(tr["final_category"].astype(str))
    y_te = le_cat.transform(te["final_category"].astype(str))

    clf = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ])

    clf.fit(tr["text"], y_tr)
    yhat = clf.predict(te["text"])
    m = f1_score(y_te, yhat, average="macro")
    summary_metrics["classification_macro_f1"] = float(m)

    out_dir = f"{ART}/classification_tfidf_lr"
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(clf, f"{out_dir}/model.joblib")
    joblib.dump(le_cat, f"{out_dir}/label_encoder.joblib")
    print("Fallback Classification (TF-IDF+LR) Macro-F1:", round(m, 4))

else:
    # -------------------------------
    # Tiny-data mode: no training
    # -------------------------------
    summary_metrics["classification_macro_f1"] = None
    print(
        f"Skipped supervised classification (labeled rows={len(cat_df)}). "
        f"Use rule-based/zero-shot inference fallback at runtime."
    )

Category rows: 1398 | CUDA: False | XLA: True | fp16: False


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Classification Macro-F1: 0.0706


In [ ]:
# ============================================================
# 2) PRIORITY MODEL — LightGBM multiclass P1..P5
# ============================================================
prio_df = df[df["final_priority"].isin(list(ALLOWED_P))].copy()
if len(prio_df) > 300:
    tr, va, te = time_split(prio_df)

    for frame in [tr,va,te]:
        frame["text_merged"] = (
            frame["summary"].astype(str) + " " +
            frame["description"].astype(str) + " " +
            frame["comments_text"].astype(str) + " " +
            frame["top_error_lines"].astype(str)
        )

    num_cols = ["affected_users","downtime_minutes","error_count","fatal_count","timeout_count","auth_error_count"]
    cat_cols = ["env","service"]

    le_p = LabelEncoder()
    ytr = le_p.fit_transform(tr["final_priority"])
    yte = le_p.transform(te["final_priority"])

    pre = ColumnTransformer([
        ("txt", TfidfVectorizer(max_features=50000, ngram_range=(1,2), min_df=3), "text_merged"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        ("num", "passthrough", num_cols),
    ])

    clf = lgb.LGBMClassifier(
        objective="multiclass", num_class=5,
        learning_rate=0.05, num_leaves=63,
        feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
        min_data_in_leaf=50, reg_alpha=1.0, reg_lambda=2.0,
        n_estimators=700, random_state=42
    )

    pipe = Pipeline([("pre", pre), ("clf", clf)])
    pipe.fit(tr[["text_merged"]+cat_cols+num_cols], ytr)
    pred = pipe.predict(te[["text_merged"]+cat_cols+num_cols])

    macro = f1_score(yte, pred, average="macro")
    canon = ["P1","P2","P3","P4","P5"]
    ord_map = {p:i for i,p in enumerate(canon)}
    true_ord = te["final_priority"].map(ord_map).values
    pred_lbl = le_p.inverse_transform(pred)
    pred_ord = pd.Series(pred_lbl).map(ord_map).values
    off1 = float(np.mean(np.abs(true_ord - pred_ord) <= 1))

    summary_metrics["priority_macro_f1"] = float(macro)
    summary_metrics["priority_off_by_1"] = off1

    out_dir = f"{ART}/priority_lgbm"
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(pipe, f"{out_dir}/model.joblib")
    joblib.dump(le_p, f"{out_dir}/label_encoder.joblib")
    print("Priority Macro-F1:", round(macro,4), "| Off-by-1:", round(off1,4))
else:
    print("Skipped priority (insufficient labeled rows).")

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Info] Auto-choosing row-wise multi-threading, 

In [ ]:
# # ============================================================
# # 3) EMOTION MODEL — DistilRoBERTa
# # ============================================================
# emo_labels = {"angry","frustrated","urgent","neutral"}
# emo_df = df[df["final_emotion"].astype(str).str.lower().isin(emo_labels)].copy()
# if len(emo_df) > 200:
#     emo_df["final_emotion"] = emo_df["final_emotion"].str.lower()
#     tr, va, te = time_split(emo_df)

#     def mk_text2(x):
#         return x["summary"] + " [SEP] " + x["description"] + " [SEP] " + x["comments_text"]

#     tr["text"] = tr.apply(mk_text2, axis=1)
#     va["text"] = va.apply(mk_text2, axis=1)
#     te["text"] = te.apply(mk_text2, axis=1)

#     le_e = LabelEncoder()
#     ytr = le_e.fit_transform(tr["final_emotion"])
#     yva = le_e.transform(va["final_emotion"])
#     yte = le_e.transform(te["final_emotion"])

#     ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
#     ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
#     ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

#     mname = "distilroberta-base"
#     tok = AutoTokenizer.from_pretrained(mname)
#     def tf(b): return tok(b["text"], truncation=True, max_length=256)
#     ds_tr = ds_tr.map(tf, batched=True)
#     ds_va = ds_va.map(tf, batched=True)
#     ds_te = ds_te.map(tf, batched=True)

#     model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=len(le_e.classes_))

#     def comp2(ep):
#         logits, labels = ep
#         p = np.argmax(logits, axis=-1)
#         return {"macro_f1": f1_score(labels, p, average="macro"), "accuracy": accuracy_score(labels,p)}

#     out_dir = f"{ART}/emotion_distilroberta"
#     args = TrainingArguments(
#         output_dir=out_dir,
#         eval_strategy="epoch",
#         save_strategy="epoch",
#         learning_rate=3e-5,
#         per_device_train_batch_size=16,
#         per_device_eval_batch_size=32,
#         num_train_epochs=4,
#         warmup_ratio=0.06,
#         weight_decay=0.01,
#         fp16=torch.cuda.is_available(),
#         load_best_model_at_end=True,
#         metric_for_best_model="macro_f1",
#         save_total_limit=2,
#         report_to="none"
#     )
#     trainer = Trainer(
#         model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
#         tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=comp2
#     )
#     trainer.train()
#     pred = trainer.predict(ds_te)
#     yhat = np.argmax(pred.predictions, axis=-1)
#     macro = f1_score(yte, yhat, average="macro")
#     summary_metrics["emotion_macro_f1"] = float(macro)

#     trainer.save_model(out_dir); tok.save_pretrained(out_dir); joblib.dump(le_e, f"{out_dir}/label_encoder.joblib")
#     print("Emotion Macro-F1:", round(macro,4))
# else:
#     print("Skipped emotion (insufficient labeled rows).")

# ============================================================
# 3) EMOTION MODEL — DistilRoBERTa (updated)
# ============================================================

emo_labels = {"angry", "frustrated", "urgent", "neutral"}
emo_df = df[df["final_emotion"].astype(str).str.lower().isin(emo_labels)].copy()

# Reuse helpers if already defined above; otherwise define them here.
def is_xla_runtime():
    try:
        import torch_xla  # noqa: F401
        return True
    except Exception:
        return False

USE_FP16 = torch.cuda.is_available() and not is_xla_runtime()
USE_BF16 = False

def build_trainer(model, args, train_dataset, eval_dataset, tok, compute_metrics):
    return Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=compute_metrics
    )

def build_args(out_dir, train_bs=16, eval_bs=32, epochs=4, lr=3e-5, metric="macro_f1"):
    return TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=lr,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        num_train_epochs=epochs,
        warmup_steps=10,
        weight_decay=0.01,
        fp16=USE_FP16,
        bf16=USE_BF16,
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model=metric,
        save_total_limit=2,
        report_to="none"
    )

if len(emo_df) > 200:
    emo_df["final_emotion"] = emo_df["final_emotion"].astype(str).str.lower()
    tr, va, te = time_split(emo_df)

    def mk_text2(x):
        return (
            str(x.get("summary", "")) + " [SEP] " +
            str(x.get("description", "")) + " [SEP] " +
            str(x.get("comments_text", ""))
        )

    tr["text"] = tr.apply(mk_text2, axis=1)
    va["text"] = va.apply(mk_text2, axis=1)
    te["text"] = te.apply(mk_text2, axis=1)

    le_e = LabelEncoder()
    ytr = le_e.fit_transform(tr["final_emotion"])
    yva = le_e.transform(va["final_emotion"])
    yte = le_e.transform(te["final_emotion"])

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

    mname = "distilroberta-base"
    tok = AutoTokenizer.from_pretrained(mname)

    def tf(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    ds_tr = ds_tr.map(tf, batched=True)
    ds_va = ds_va.map(tf, batched=True)
    ds_te = ds_te.map(tf, batched=True)

    model = AutoModelForSequenceClassification.from_pretrained(
        mname,
        num_labels=len(le_e.classes_)
    )

    def comp2(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/emotion_distilroberta"
    args = build_args(
        out_dir=out_dir,
        train_bs=16,
        eval_bs=32,
        epochs=4,
        lr=3e-5,
        metric="macro_f1"
    )

    trainer = build_trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        tok=tok,
        compute_metrics=comp2
    )

    trainer.train()
    pred = trainer.predict(ds_te)
    yhat = np.argmax(pred.predictions, axis=-1)
    macro = f1_score(yte, yhat, average="macro")
    summary_metrics["emotion_macro_f1"] = float(macro)

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    joblib.dump(le_e, f"{out_dir}/label_encoder.joblib")

    print("Emotion Macro-F1:", round(macro, 4))
else:
    print("Skipped emotion (insufficient labeled rows).")

Map:   0%|          | 0/978 [00:00<?, ? examples/s]

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

Map:   0%|          | 0/210 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: distilroberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent

...

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Emotion Macro-F1: 0.8369


In [ ]:
# # ============================================================
# # 4) LOG RELEVANCE RERANKER — MiniLM pair classifier
# # ============================================================
# # Needs row-level pairs: ticket_context, log_line, log_relevant (0/1)
# log_df = df[(df["ticket_context"].str.len()>0) & (df["log_line"].str.len()>0)].copy()
# if len(log_df) > 500 and log_df["log_relevant"].nunique() > 1:
#     tr, va, te = time_split(log_df)

#     def pair_text(x):
#         return x["ticket_context"] + " [SEP] " + x["log_line"]

#     tr["text"] = tr.apply(pair_text, axis=1)
#     va["text"] = va.apply(pair_text, axis=1)
#     te["text"] = te.apply(pair_text, axis=1)

#     ytr = tr["log_relevant"].astype(int).values
#     yva = va["log_relevant"].astype(int).values
#     yte = te["log_relevant"].astype(int).values

#     ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
#     ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
#     ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

#     mname = "sentence-transformers/all-MiniLM-L6-v2"
#     tok = AutoTokenizer.from_pretrained(mname)
#     model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2)

#     def tf(b): return tok(b["text"], truncation=True, max_length=256)
#     ds_tr = ds_tr.map(tf, batched=True); ds_va = ds_va.map(tf, batched=True); ds_te = ds_te.map(tf, batched=True)

#     def comp3(ep):
#         logits, labels = ep
#         pred = np.argmax(logits, axis=-1)
#         return {"macro_f1": f1_score(labels, pred, average="macro"), "accuracy": accuracy_score(labels,pred)}

#     out_dir = f"{ART}/log_reranker_minilm"
#     args = TrainingArguments(
#         output_dir=out_dir, eval_strategy="epoch", save_strategy="epoch",
#         learning_rate=2e-5, per_device_train_batch_size=32, per_device_eval_batch_size=64,
#         num_train_epochs=3, fp16=torch.cuda.is_available(), load_best_model_at_end=True,
#         metric_for_best_model="macro_f1", report_to="none"
#     )
#     trainer = Trainer(
#         model=model, args=args, train_dataset=ds_tr, eval_dataset=ds_va,
#         tokenizer=tok, data_collator=DataCollatorWithPadding(tok), compute_metrics=comp3
#     )
#     trainer.train()

#     pred = trainer.predict(ds_te)
#     logits = pred.predictions
#     probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()

#     # Precision@20 approximation (global)
#     idx = np.argsort(-probs)[:20]
#     p20 = float(np.mean(yte[idx])) if len(idx)>0 else 0.0
#     yhat = (probs >= 0.5).astype(int)
#     macro = f1_score(yte, yhat, average="macro")

#     summary_metrics["log_rerank_macro_f1"] = float(macro)
#     summary_metrics["log_rerank_precision_at_20"] = p20

#     trainer.save_model(out_dir); tok.save_pretrained(out_dir)
#     print("Log Reranker Macro-F1:", round(macro,4), "| P@20:", round(p20,4))
# else:
#     print("Skipped log reranker (need ticket_context, log_line, log_relevant with enough rows).")

# ============================================================
# 4) LOG RELEVANCE RERANKER — MiniLM pair classifier
# With fallback to rule-based and weak supervision
# ============================================================

print("\n=== LOG RELEVANCE RERANKER ===")

log_df = df[(df["ticket_context"].astype(str).str.len() > 0) &
            (df["log_line"].astype(str).str.len() > 0)].copy()

print(f"Paired rows (ticket_context + log_line): {len(log_df)}")
print(f"log_relevant unique values: {log_df['log_relevant'].nunique() if len(log_df) > 0 else 0}")

if len(log_df) > 0:
    print(f"log_relevant distribution:\n{log_df['log_relevant'].value_counts(dropna=False)}")

# ----------------------------
# Option 1: Full supervised training (>500 pairs with both labels)
# ----------------------------
if len(log_df) > 500 and log_df["log_relevant"].nunique() > 1:
    print("\n✓ Sufficient labeled pairs. Training MiniLM reranker...")

    tr, va, te = time_split(log_df)

    def pair_text(x):
        return str(x["ticket_context"]) + " [SEP] " + str(x["log_line"])

    tr["text"] = tr.apply(pair_text, axis=1)
    va["text"] = va.apply(pair_text, axis=1)
    te["text"] = te.apply(pair_text, axis=1)

    ytr = tr["log_relevant"].astype(int).values
    yva = va["log_relevant"].astype(int).values
    yte = te["log_relevant"].astype(int).values

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

    mname = "sentence-transformers/all-MiniLM-L6-v2"
    tok = AutoTokenizer.from_pretrained(mname)
    model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2)

    def tf(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    ds_tr = ds_tr.map(tf, batched=True)
    ds_va = ds_va.map(tf, batched=True)
    ds_te = ds_te.map(tf, batched=True)

    def comp3(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/log_reranker_minilm"
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=3,
        warmup_steps=5,
        fp16=torch.cuda.is_available() and not is_xla_runtime(),
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="none",
        logging_steps=50
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=comp3
    )

    trainer.train()

    pred = trainer.predict(ds_te)
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()

    # Precision@20 approximation
    idx = np.argsort(-probs)[:20]
    p20 = float(np.mean(yte[idx])) if len(idx) > 0 else 0.0
    yhat = (probs >= 0.5).astype(int)
    macro = f1_score(yte, yhat, average="macro")

    summary_metrics["log_rerank_macro_f1"] = float(macro)
    summary_metrics["log_rerank_precision_at_20"] = p20
    summary_metrics["log_rerank_mode"] = "supervised_minilm"

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    print(f"✓ Log Reranker Macro-F1: {round(macro, 4)} | P@20: {round(p20, 4)}")

# ----------------------------
# Option 2: Weak supervision (30-500 pairs or imbalanced labels)
# ----------------------------
elif len(log_df) > 30 and log_df["log_relevant"].nunique() > 1:
    print("\n⚠ Limited labeled pairs. Training with weak supervision...")

    tr, va, te = time_split(log_df)

    def pair_text(x):
        return str(x["ticket_context"]) + " [SEP] " + str(x["log_line"])

    tr["text"] = tr.apply(pair_text, axis=1)
    va["text"] = va.apply(pair_text, axis=1)
    te["text"] = te.apply(pair_text, axis=1)

    ytr = tr["log_relevant"].astype(int).values
    yva = va["log_relevant"].astype(int).values
    yte = te["log_relevant"].astype(int).values

    ds_tr = Dataset.from_pandas(pd.DataFrame({"text": tr["text"], "label": ytr}))
    ds_va = Dataset.from_pandas(pd.DataFrame({"text": va["text"], "label": yva}))
    ds_te = Dataset.from_pandas(pd.DataFrame({"text": te["text"], "label": yte}))

    mname = "sentence-transformers/all-MiniLM-L6-v2"
    tok = AutoTokenizer.from_pretrained(mname)
    model = AutoModelForSequenceClassification.from_pretrained(mname, num_labels=2)

    def tf(batch):
        return tok(batch["text"], truncation=True, max_length=256)

    ds_tr = ds_tr.map(tf, batched=True)
    ds_va = ds_va.map(tf, batched=True)
    ds_te = ds_te.map(tf, batched=True)

    def comp3(eval_pred):
        logits, labels = eval_pred
        pred = np.argmax(logits, axis=-1)
        return {
            "macro_f1": f1_score(labels, pred, average="macro"),
            "accuracy": accuracy_score(labels, pred)
        }

    out_dir = f"{ART}/log_reranker_minilm_weak"
    args = TrainingArguments(
        output_dir=out_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=2,
        warmup_steps=2,
        weight_decay=0.01,
        fp16=torch.cuda.is_available() and not is_xla_runtime(),
        optim="adamw_torch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        report_to="none",
        logging_steps=10
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tr,
        eval_dataset=ds_va,
        data_collator=DataCollatorWithPadding(tokenizer=tok),
        compute_metrics=comp3
    )

    trainer.train()

    pred = trainer.predict(ds_te)
    logits = pred.predictions
    probs = torch.softmax(torch.tensor(logits, dtype=torch.float32), dim=-1)[:, 1].numpy()

    idx = np.argsort(-probs)[:min(20, len(probs))]
    p20 = float(np.mean(yte[idx])) if len(idx) > 0 else 0.0
    yhat = (probs >= 0.5).astype(int)
    macro = f1_score(yte, yhat, average="macro", zero_division=0)

    summary_metrics["log_rerank_macro_f1"] = float(macro)
    summary_metrics["log_rerank_precision_at_20"] = p20
    summary_metrics["log_rerank_mode"] = "weak_supervision_minilm"

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    print(f"⚠ Log Reranker (weak) Macro-F1: {round(macro, 4)} | P@20: {round(p20, 4)}")

# ----------------------------
# Option 3: Rule-based fallback (no sufficient labels)
# ----------------------------
else:
    print(f"\n✗ Insufficient labeled pairs ({len(log_df)} rows with <2 label classes).")
    print("  Falling back to rule-based log reranker...")

    def rule_based_log_relevance(ticket_context, log_line):
        """
        Simple heuristic relevance score based on keyword/token overlap
        and error signal detection.
        """
        context = str(ticket_context).lower()
        log = str(log_line).lower()

        score = 0

        # Exact token overlap
        context_tokens = set(context.split())
        log_tokens = set(log.split())
        overlap = len(context_tokens & log_tokens)
        score += overlap

        # Error/exception/failure signals
        error_signals = ["error", "exception", "failed", "failure", "fault", "crash"]
        for signal in error_signals:
            if signal in log:
                score += 2

        # Timeout signals
        if "timeout" in log or "timed out" in log:
            score += 2

        # Authentication/authorization signals
        if "auth" in log or "unauthorized" in log or "forbidden" in log:
            score += 2

        # Stack trace or traceback
        if "traceback" in log or "stack" in log:
            score += 1

        return float(score)

    # Save rule-based reranker config
    out_dir = f"{ART}/log_reranker_rulebased"
    os.makedirs(out_dir, exist_ok=True)

    rule_config = {
        "type": "rule_based",
        "signals": [
            "token_overlap",
            "error_keywords",
            "timeout_keywords",
            "auth_keywords",
            "stack_trace_indicators"
        ],
        "reason": f"Insufficient labeled pairs (found {len(log_df)} rows). Using heuristic scoring."
    }

    with open(f"{out_dir}/config.json", "w") as f:
        json.dump(rule_config, f, indent=2)

    summary_metrics["log_rerank_macro_f1"] = None
    summary_metrics["log_rerank_precision_at_20"] = None
    summary_metrics["log_rerank_mode"] = "rule_based_fallback"

    print(f"✓ Rule-based reranker configured. Config saved to {out_dir}/config.json")
    print(f"  This will be used in inference to score log relevance heuristically.")

print("=== LOG RELEVANCE RERANKER COMPLETE ===\n")


=== LOG RELEVANCE RERANKER ===
Paired rows (ticket_context + log_line): 0
log_relevant unique values: 0

✗ Insufficient labeled pairs (0 rows with <2 label classes).
  Falling back to rule-based log reranker...
✓ Rule-based reranker configured. Config saved to /content/drive/MyDrive/jsm_ai_ops/artifacts_all/log_reranker_rulebased/config.json
  This will be used in inference to score log relevance heuristically.
=== LOG RELEVANCE RERANKER COMPLETE ===



In [ ]:
# ============================================================
# 5) RETRIEVAL EVAL — BGE + FAISS (Recall@K, MRR@10)
# ============================================================
# Needs: retrieval_query, retrieval_doc, retrieval_group_id, retrieval_relevant (0/1)
ret_df = df[(df["retrieval_query"].str.len()>0) & (df["retrieval_doc"].str.len()>0)].copy()
if len(ret_df) > 500 and ret_df["retrieval_relevant"].nunique() > 1:
    # use test slice only to evaluate retrieval behavior
    _, _, te = time_split(ret_df)

    embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

    docs = te["retrieval_doc"].astype(str).tolist()
    doc_emb = embed_model.encode(docs, normalize_embeddings=True, show_progress_bar=True)
    doc_emb = np.asarray(doc_emb, dtype="float32")

    index = faiss.IndexFlatIP(doc_emb.shape[1])
    index.add(doc_emb)

    # group by query id
    groups = te.groupby("retrieval_group_id", dropna=False)
    recall_at_5_list = []
    mrr10_list = []

    # map row->global doc idx
    te = te.reset_index(drop=True)
    # query once per group using first query text
    for gid, g in groups:
        qtext = str(g["retrieval_query"].iloc[0])
        qemb = embed_model.encode([qtext], normalize_embeddings=True)
        qemb = np.asarray(qemb, dtype="float32")

        D, I = index.search(qemb, 10)
        ranked_idx = I[0].tolist()

        # relevant doc indices in global te frame
        rel_idx = set(g[g["retrieval_relevant"].astype(int)==1].index.tolist())

        top5 = set(ranked_idx[:5])
        hit = 1.0 if len(rel_idx & top5) > 0 else 0.0
        recall_at_5_list.append(hit)

        rr = 0.0
        for rank, di in enumerate(ranked_idx, start=1):
            if di in rel_idx:
                rr = 1.0/rank
                break
        mrr10_list.append(rr)

    r5 = float(np.mean(recall_at_5_list)) if recall_at_5_list else 0.0
    mrr10 = float(np.mean(mrr10_list)) if mrr10_list else 0.0

    summary_metrics["retrieval_recall_at_5"] = r5
    summary_metrics["retrieval_mrr_at_10"] = mrr10

    out_dir = f"{ART}/retrieval_bge"
    os.makedirs(out_dir, exist_ok=True)
    faiss.write_index(index, f"{out_dir}/faiss.index")
    with open(f"{out_dir}/metrics.json", "w") as f:
        json.dump({"recall_at_5": r5, "mrr_at_10": mrr10}, f, indent=2)

    print("Retrieval Recall@5:", round(r5,4), "| MRR@10:", round(mrr10,4))
else:
    print("Skipped retrieval eval (need retrieval_* columns with enough rows).")

Skipped retrieval eval (need retrieval_* columns with enough rows).


In [ ]:
# ============================================================
# 6) INCIDENT RISK — LightGBM binary
# ============================================================
risk_df = df[df["incident_in_next_30m"].isin([0,1])].copy()
if len(risk_df) > 400 and risk_df["incident_in_next_30m"].nunique() > 1:
    tr, va, te = time_split(risk_df)

    feat_cols = ["error_count","timeout_count","fatal_count","affected_users","downtime_minutes","auth_error_count"]
    for c in feat_cols:
        if c not in risk_df.columns:
            risk_df[c] = 0

    # include text tfidf + numeric + categorical
    for frame in [tr,va,te]:
        frame["text_merged"] = (frame["summary"].astype(str) + " " + frame["description"].astype(str) +
                                " " + frame["comments_text"].astype(str) + " " + frame["top_error_lines"].astype(str))

    pre = ColumnTransformer([
        ("txt", TfidfVectorizer(max_features=30000, ngram_range=(1,2), min_df=3), "text_merged"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), ["env","service"]),
        ("num", "passthrough", feat_cols),
    ])

    clf = lgb.LGBMClassifier(
        objective="binary",
        learning_rate=0.03,
        num_leaves=127,
        min_data_in_leaf=100,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=1,
        reg_alpha=1.0,
        reg_lambda=2.0,
        n_estimators=1000,
        random_state=42
    )

    pipe = Pipeline([("pre", pre), ("clf", clf)])
    ytr = tr["incident_in_next_30m"].astype(int).values
    yte = te["incident_in_next_30m"].astype(int).values

    pipe.fit(tr[["text_merged","env","service"]+feat_cols], ytr)
    prob = pipe.predict_proba(te[["text_merged","env","service"]+feat_cols])[:,1]
    pr_auc = float(average_precision_score(yte, prob))

    # Recall at top alert budget (example top 20/day approximated by top 5% samples here)
    k = max(1, int(0.05 * len(prob)))
    idx = np.argsort(-prob)[:k]
    recall_top = float(yte[idx].sum() / max(1, yte.sum()))

    summary_metrics["incident_pr_auc"] = pr_auc
    summary_metrics["incident_recall_top5pct"] = recall_top

    out_dir = f"{ART}/incident_lgbm"
    os.makedirs(out_dir, exist_ok=True)
    joblib.dump(pipe, f"{out_dir}/model.joblib")
    with open(f"{out_dir}/metrics.json","w") as f:
        json.dump({"pr_auc": pr_auc, "recall_top5pct": recall_top}, f, indent=2)

    print("Incident PR-AUC:", round(pr_auc,4), "| Recall@Top5%:", round(recall_top,4))
else:
    print("Skipped incident model (need incident_in_next_30m binary labels and enough rows).")

Skipped incident model (need incident_in_next_30m binary labels and enough rows).


In [ ]:
# ============================================================
# 7) FINAL INFERENCE + RCA PIPELINE
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import torch

# Optional imports used only if corresponding artifacts/models exist
try:
    import faiss
except Exception:
    faiss = None

try:
    from sentence_transformers import SentenceTransformer
except Exception:
    SentenceTransformer = None

from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ----------------------------
# Paths
# ----------------------------
BASE = "/content/drive/MyDrive/jsm_ai_ops"
ART = f"{BASE}/artifacts_all"

# ----------------------------
# Helpers
# ----------------------------
def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x)

def make_ticket_text(ticket):
    return (
        safe_str(ticket.get("summary")) + " [SEP] " +
        safe_str(ticket.get("description")) + " [SEP] " +
        safe_str(ticket.get("comments_text")) + " [SEP] " +
        safe_str(ticket.get("top_error_lines"))
    )

def make_emotion_text(ticket):
    return (
        safe_str(ticket.get("summary")) + " [SEP] " +
        safe_str(ticket.get("description")) + " [SEP] " +
        safe_str(ticket.get("comments_text"))
    )

def make_priority_text(ticket):
    return (
        safe_str(ticket.get("summary")) + " " +
        safe_str(ticket.get("description")) + " " +
        safe_str(ticket.get("comments_text")) + " " +
        safe_str(ticket.get("top_error_lines"))
    )

def softmax_np(x):
    x = np.array(x, dtype=np.float64)
    x = x - np.max(x)
    e = np.exp(x)
    return e / np.sum(e)

def infer_transformer_label(text, model_dir):
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.eval()

    enc = tok(text, truncation=True, max_length=384, return_tensors="pt")
    with torch.no_grad():
        out = model(**enc)
        logits = out.logits.cpu().numpy()[0]
    probs = softmax_np(logits)
    pred_idx = int(np.argmax(probs))
    return pred_idx, probs

# ----------------------------
# Load artifacts if present
# ----------------------------
artifacts = {}

# Category
if os.path.exists(f"{ART}/classification_deberta"):
    artifacts["category_mode"] = "deberta"
    artifacts["category_model_dir"] = f"{ART}/classification_deberta"
    artifacts["category_label_encoder"] = joblib.load(f"{ART}/classification_deberta/label_encoder.joblib")
elif os.path.exists(f"{ART}/classification_tfidf_lr/model.joblib"):
    artifacts["category_mode"] = "tfidf_lr"
    artifacts["category_model"] = joblib.load(f"{ART}/classification_tfidf_lr/model.joblib")
    artifacts["category_label_encoder"] = joblib.load(f"{ART}/classification_tfidf_lr/label_encoder.joblib")

# Priority
if os.path.exists(f"{ART}/priority_lgbm/model.joblib"):
    artifacts["priority_model"] = joblib.load(f"{ART}/priority_lgbm/model.joblib")
    artifacts["priority_label_encoder"] = joblib.load(f"{ART}/priority_lgbm/label_encoder.joblib")

# Emotion
if os.path.exists(f"{ART}/emotion_distilroberta"):
    emo_le_path = f"{ART}/emotion_distilroberta/label_encoder.joblib"
    if os.path.exists(emo_le_path):
        artifacts["emotion_model_dir"] = f"{ART}/emotion_distilroberta"
        artifacts["emotion_label_encoder"] = joblib.load(emo_le_path)

# Incident
if os.path.exists(f"{ART}/incident_lgbm/model.joblib"):
    artifacts["incident_model"] = joblib.load(f"{ART}/incident_lgbm/model.joblib")

# Retrieval
if faiss is not None and os.path.exists(f"{ART}/retrieval_bge/faiss.index"):
    artifacts["retrieval_index"] = faiss.read_index(f"{ART}/retrieval_bge/faiss.index")

print("Loaded artifacts:", list(artifacts.keys()))

# ----------------------------
# Prediction functions
# ----------------------------
def predict_category(ticket):
    if "category_mode" not in artifacts:
        return {"label": None, "confidence": None, "source": "missing_model"}

    text = make_ticket_text(ticket)

    if artifacts["category_mode"] == "deberta":
        pred_idx, probs = infer_transformer_label(text, artifacts["category_model_dir"])
        label = artifacts["category_label_encoder"].inverse_transform([pred_idx])[0]
        return {
            "label": label,
            "confidence": float(np.max(probs)),
            "source": "deberta"
        }

    if artifacts["category_mode"] == "tfidf_lr":
        clf = artifacts["category_model"]
        le = artifacts["category_label_encoder"]
        pred_idx = clf.predict([text])[0]
        label = le.inverse_transform([pred_idx])[0]

        confidence = None
        if hasattr(clf, "predict_proba"):
            probs = clf.predict_proba([text])[0]
            confidence = float(np.max(probs))

        return {
            "label": label,
            "confidence": confidence,
            "source": "tfidf_lr"
        }

def predict_priority(ticket):
    if "priority_model" not in artifacts:
        return {"label": None, "confidence": None, "source": "missing_model"}

    num_cols = [
        "affected_users", "downtime_minutes", "error_count",
        "fatal_count", "timeout_count", "auth_error_count"
    ]
    cat_cols = ["env", "service"]

    row = {
        "text_merged": make_priority_text(ticket),
        "env": safe_str(ticket.get("env", "unknown")),
        "service": safe_str(ticket.get("service", "unknown")),
    }
    for c in num_cols:
        row[c] = float(ticket.get(c, 0) or 0)

    X = pd.DataFrame([row])
    pipe = artifacts["priority_model"]
    le = artifacts["priority_label_encoder"]

    pred_idx = pipe.predict(X)[0]
    label = le.inverse_transform([pred_idx])[0]

    confidence = None
    if hasattr(pipe, "predict_proba"):
        probs = pipe.predict_proba(X)[0]
        confidence = float(np.max(probs))

    return {
        "label": label,
        "confidence": confidence,
        "source": "lightgbm"
    }

def predict_emotion(ticket):
    if "emotion_model_dir" not in artifacts:
        return {"label": None, "confidence": None, "source": "missing_model"}

    text = make_emotion_text(ticket)
    pred_idx, probs = infer_transformer_label(text, artifacts["emotion_model_dir"])
    label = artifacts["emotion_label_encoder"].inverse_transform([pred_idx])[0]

    return {
        "label": label,
        "confidence": float(np.max(probs)),
        "source": "distilroberta"
    }

def predict_incident_risk(ticket):
    if "incident_model" not in artifacts:
        return {"score": None, "source": "missing_model"}

    feat_cols = [
        "error_count", "timeout_count", "fatal_count",
        "affected_users", "downtime_minutes", "auth_error_count"
    ]

    row = {
        "text_merged": make_priority_text(ticket),
        "env": safe_str(ticket.get("env", "unknown")),
        "service": safe_str(ticket.get("service", "unknown")),
    }
    for c in feat_cols:
        row[c] = float(ticket.get(c, 0) or 0)

    X = pd.DataFrame([row])
    pipe = artifacts["incident_model"]
    score = float(pipe.predict_proba(X)[0][1])

    return {
        "score": score,
        "source": "lightgbm"
    }

# ----------------------------
# Optional log reranking (rule-based placeholder if no model)
# ----------------------------
def rank_logs(ticket, candidate_logs, top_k=5):
    text = make_ticket_text(ticket).lower()

    scored = []
    for log in candidate_logs:
        log_s = safe_str(log)
        score = 0

        # simple lexical overlap heuristic
        for tok in set(text.split()):
            if len(tok) > 3 and tok in log_s.lower():
                score += 1

        if "error" in log_s.lower():
            score += 1
        if "exception" in log_s.lower():
            score += 1
        if "timeout" in log_s.lower():
            score += 1
        if "failed" in log_s.lower():
            score += 1

        scored.append((log_s, score))

    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    return [{"log_line": s[0], "score": float(s[1])} for s in scored[:top_k]]

# ----------------------------
# Optional retrieval placeholder
# ----------------------------
def retrieve_docs(ticket, doc_df=None, top_k=5):
    if doc_df is None or len(doc_df) == 0:
        return []

    query = make_ticket_text(ticket).lower()
    rows = []

    for _, r in doc_df.iterrows():
        doc = safe_str(r.get("retrieval_doc", ""))
        score = 0
        for tok in set(query.split()):
            if len(tok) > 3 and tok in doc.lower():
                score += 1
        rows.append((doc, score))

    rows = sorted(rows, key=lambda x: x[1], reverse=True)
    return [{"doc": x[0], "score": float(x[1])} for x in rows[:top_k]]

# ----------------------------
# RCA generation
# ----------------------------
def generate_rca(ticket, result_bundle):
    findings = []

    category = result_bundle.get("category", {}).get("label")
    priority = result_bundle.get("priority", {}).get("label")
    emotion = result_bundle.get("emotion", {}).get("label")
    risk = result_bundle.get("incident_risk", {}).get("score")

    err = float(ticket.get("error_count", 0) or 0)
    fatal = float(ticket.get("fatal_count", 0) or 0)
    timeout = float(ticket.get("timeout_count", 0) or 0)
    auth = float(ticket.get("auth_error_count", 0) or 0)
    affected = float(ticket.get("affected_users", 0) or 0)
    downtime = float(ticket.get("downtime_minutes", 0) or 0)
    env = safe_str(ticket.get("env", "unknown"))
    service = safe_str(ticket.get("service", "unknown"))

    likely_causes = []

    if auth > 0:
        likely_causes.append("authentication or authorization failure")
    if timeout > 0:
        likely_causes.append("timeout or downstream service latency")
    if fatal > 0:
        likely_causes.append("fatal application/runtime failure")
    if err > 50:
        likely_causes.append("high error-volume service degradation")
    if env.lower() in {"prod", "production"} and downtime > 0:
        likely_causes.append("production service disruption")

    if not likely_causes and category:
        likely_causes.append(f"issue related to category '{category}'")

    findings.append(f"Service: {service or 'unknown'} in env: {env or 'unknown'}.")
    if priority:
        findings.append(f"Predicted priority is {priority}.")
    if emotion:
        findings.append(f"Reporter/user emotion appears {emotion}.")
    if risk is not None:
        findings.append(f"Predicted near-term incident risk score is {risk:.3f}.")
    if affected > 0:
        findings.append(f"Estimated affected users: {int(affected)}.")
    if downtime > 0:
        findings.append(f"Observed downtime minutes: {int(downtime)}.")

    top_logs = result_bundle.get("top_logs", [])
    if top_logs:
        findings.append("Most relevant log evidence:")
        for item in top_logs[:3]:
            findings.append(f"- {item['log_line']}")

    retrieved = result_bundle.get("retrieved_docs", [])
    if retrieved:
        findings.append("Most relevant retrieved references:")
        for item in retrieved[:2]:
            findings.append(f"- {item['doc'][:200]}")

    summary = {
        "likely_root_causes": likely_causes,
        "evidence_summary": findings,
        "recommended_next_steps": [
            "Inspect the top reranked log lines for the first failing component.",
            "Check recent deployments/config changes for the predicted service/category.",
            "Validate dependency health and timeout/authentication paths.",
            "Compare with retrieved similar incidents or support documents."
        ]
    }
    return summary

# ----------------------------
# Unified inference
# ----------------------------
def predict_ticket(ticket, candidate_logs=None, retrieval_df=None):
    if candidate_logs is None:
        candidate_logs = []

    category = predict_category(ticket)
    priority = predict_priority(ticket)
    emotion = predict_emotion(ticket)
    incident_risk = predict_incident_risk(ticket)
    top_logs = rank_logs(ticket, candidate_logs, top_k=5)
    retrieved_docs = retrieve_docs(ticket, retrieval_df, top_k=5)

    bundle = {
        "category": category,
        "priority": priority,
        "emotion": emotion,
        "incident_risk": incident_risk,
        "top_logs": top_logs,
        "retrieved_docs": retrieved_docs
    }

    bundle["rca"] = generate_rca(ticket, bundle)
    return bundle

# ============================================================
# 8) EXAMPLE USAGE
# ============================================================

sample_ticket = {
    "summary": "Users cannot log in to production dashboard",
    "description": "Multiple customers report login failures after this morning deployment. API returns 401 and some requests timeout.",
    "comments_text": "Escalated by support. Issue seems widespread.",
    "top_error_lines": "AuthServiceError: token validation failed; TimeoutError: upstream request timed out",
    "affected_users": 120,
    "downtime_minutes": 25,
    "error_count": 180,
    "fatal_count": 3,
    "timeout_count": 42,
    "auth_error_count": 85,
    "env": "production",
    "service": "auth-service"
}

sample_logs = [
    "INFO startup complete",
    "ERROR token validation failed for issuer mismatch",
    "WARN upstream request timed out after 30s",
    "ERROR login handler exception in auth-service",
    "INFO cache refresh complete"
]

# optional retrieval dataframe from your combined df
retrieval_df = None
if "retrieval_doc" in df.columns and "retrieval_query" in df.columns:
    retrieval_df = df[df["retrieval_doc"].astype(str).str.len() > 0][["retrieval_doc"]].drop_duplicates().copy()

result = predict_ticket(sample_ticket, candidate_logs=sample_logs, retrieval_df=retrieval_df)

print(json.dumps(result, indent=2))

Loaded artifacts: ['category_mode', 'category_model_dir', 'category_label_encoder', 'priority_model', 'priority_label_encoder', 'emotion_model_dir', 'emotion_label_encoder']


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

{
  "category": {
    "label": "account",
    "confidence": NaN,
    "source": "deberta"
  },
  "priority": {
    "label": "P2",
    "confidence": 0.9014735599926677,
    "source": "lightgbm"
  },
  "emotion": {
    "label": "neutral",
    "confidence": 0.9921613778196002,
    "source": "distilroberta"
  },
  "incident_risk": {
    "score": null,
    "source": "missing_model"
  },
  "top_logs": [
    {
      "log_line": "ERROR token validation failed for issuer mismatch",
      "score": 5.0
    },
    {
      "log_line": "WARN upstream request timed out after 30s",
      "score": 4.0
    },
    {
      "log_line": "ERROR login handler exception in auth-service",
      "score": 3.0
    },
    {
      "log_line": "INFO startup complete",
      "score": 0.0
    },
    {
      "log_line": "INFO cache refresh complete",
      "score": 0.0
    }
  ],
  "retrieved_docs": [],
  "rca": {
    "likely_root_causes": [
      "authentication or authorization failure",
      "timeout or downstream se

In [ ]:
# ============================================================
# Save consolidated metrics
# ============================================================
with open(f"{ART}/summary_metrics.json","w") as f:
    json.dump(summary_metrics, f, indent=2)

print("\n=== SUMMARY METRICS ===")
print(json.dumps(summary_metrics, indent=2))
print(f"\nArtifacts saved under: {ART}")


=== SUMMARY METRICS ===
{
  "emotion_macro_f1": 0.8368889809444905
}

Artifacts saved under: /content/drive/MyDrive/jsm_ai_ops/artifacts_all
